# PCA — Factor Structure Analysis

## What it does
Principal Component Analysis (PCA) finds **orthogonal directions of maximum variance** in a dataset.
Given a matrix of T observations × N series, PCA extracts K latent factors:

$$X = F \Lambda^\top + E$$

where F are the factor scores (T × K), Λ are the loadings (N × K), and E is idiosyncratic noise.

## When to use it
- You want to understand the **latent factor structure** of a set of correlated time series
- You need to reduce the dimensionality of features before another model
- You want to answer: "how many independent directions exist in this data?"

## How many components?
Three standard criteria, each answering a slightly different question:
| Criterion | Rule | Interpretation |
|---|---|---|
| Kaiser | Keep components with eigenvalue > 1 | Each retained component explains more than one original variable |
| 90%/95%/99% variance | Keep enough components to reach threshold | How many factors capture X% of total variation |
| Scree plot | Elbow in eigenvalue curve | Visual — stop where the curve flattens |

## Data format
A wide matrix: rows = time periods, columns = variables (e.g. portfolio returns, macro series).
Missing values are handled by dropping sparse columns and then incomplete rows.

## Configuration

Edit the values below to adapt this notebook to a new dataset.

In [ ]:
CONFIG = {
    # --- Data ---
    # Expects a wide matrix: rows=time, columns=variables
    # First column or index should be the date.
    'DATA_FILE':            '../../data/FREDMD.csv',
    'DATE_COL':             'sasdate',
    # --- Cleaning ---
    # Drop columns with fewer than this fraction of non-null observations
    'MIN_OBS_FRAC':         0.5,
    'STANDARDIZE':          True,
    # --- PCA ---
    'N_COMPONENTS':         None,     # None = keep all; set an int to limit
    # --- Variance thresholds for factor count summary ---
    'VARIANCE_THRESHOLDS':  [0.80, 0.90, 0.95, 0.99],
    # Top N loadings to display per component
    'TOP_LOADINGS':         10,
    # Number of components to display in plots
    'N_SCREE_SHOW':         20,
    # --- Output ---
    'SAVE_RESULTS':         True,
    'OUTPUT_DIR':           '../../results',
}

print('Configuration loaded.')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

## Step 1 — Load Data & Clean

In [ ]:
import sys, warnings, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('../../').resolve()))
from utils import load_csv, load_parquet

if CONFIG['DATA_FILE'].endswith(('.pq', '.parquet')):
    df_raw = load_parquet(CONFIG['DATA_FILE'])
else:
    df_raw = load_csv(CONFIG['DATA_FILE'])

# Set date as index if present
if CONFIG['DATE_COL'] in df_raw.columns:
    df_raw = df_raw.set_index(CONFIG['DATE_COL'])

print(f'Raw shape : {df_raw.shape}')
print(f'Period    : {df_raw.index[0]} — {df_raw.index[-1]}')
df_raw.head()

In [ ]:
# Keep only numeric columns
df_raw = df_raw.select_dtypes(include='number')

# Drop columns with too many missing values
min_obs = int(len(df_raw) * CONFIG['MIN_OBS_FRAC'])
df_clean = df_raw.dropna(axis=1, thresh=min_obs).dropna(axis=0)

print(f'After cleaning : {df_clean.shape[0]} rows × {df_clean.shape[1]} columns')
print(f'Dropped columns: {df_raw.shape[1] - df_clean.shape[1]}')
print(f'Dropped rows   : {df_raw.shape[0] - df_clean.shape[0]}')

## Step 2 — Standardize

In [ ]:
if CONFIG['STANDARDIZE']:
    scaler  = StandardScaler()
    X_scaled = scaler.fit_transform(df_clean.values)
    print('Features standardized (zero mean, unit variance).')
else:
    X_scaled = df_clean.values
    print('Standardization skipped.')

## Step 3 — Fit PCA

In [ ]:
n_components = CONFIG['N_COMPONENTS'] or min(df_clean.shape)
pca = PCA(n_components=n_components)
pca.fit(X_scaled)

eigenvalues  = pca.explained_variance_
var_ratio    = pca.explained_variance_ratio_
cum_var      = np.cumsum(var_ratio)

n_show = min(CONFIG['N_SCREE_SHOW'], n_components)
print(f'Total components : {n_components}')
print(f'\nTop-5 eigenvalues : {np.round(eigenvalues[:5], 2)}')
print(f'Top-5 var %       : {np.round(var_ratio[:5]*100, 2)}')
print(f'Cumulative at PC5 : {cum_var[min(4, n_components-1)]*100:.1f}%')

## Step 4 — Scree Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Eigenvalues
axes[0].bar(range(1, n_show+1), eigenvalues[:n_show], color='steelblue', alpha=0.8)
axes[0].axhline(y=1, color='red', linestyle='--', linewidth=1.5, label='Kaiser (EV=1)')
axes[0].set(xlabel='Principal Component', ylabel='Eigenvalue', title='Scree Plot')
axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='y')

# Cumulative variance
axes[1].plot(range(1, n_show+1), cum_var[:n_show]*100, 'b-o', linewidth=2, markersize=5)
threshold_colors = {0.80: 'orange', 0.90: 'red', 0.95: 'purple', 0.99: 'darkred'}
for thresh in CONFIG['VARIANCE_THRESHOLDS']:
    if thresh in threshold_colors:
        axes[1].axhline(y=thresh*100, color=threshold_colors[thresh],
                        linestyle='--', linewidth=1, label=f'{thresh*100:.0f}%')
        n_cross = int(np.searchsorted(cum_var, thresh)) + 1
        if n_cross <= n_show:
            axes[1].annotate(f'PC{n_cross}', (n_cross, thresh*100),
                             textcoords='offset points', xytext=(6, 3), fontsize=8,
                             color=threshold_colors[thresh])
axes[1].set(xlabel='Number of Components', ylabel='Cumulative Variance (%)',
            title='Cumulative Variance Explained')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Step 5 — Factor Count Summary

In [ ]:
n_kaiser = int(np.sum(eigenvalues > 1))
threshold_counts = {}
for t in CONFIG['VARIANCE_THRESHOLDS']:
    threshold_counts[f'{t*100:.0f}% variance'] = min(int(np.searchsorted(cum_var, t)) + 1, n_components)

summary_rows = [
    {'Criterion': 'Kaiser (eigenvalue > 1)', 'N components': n_kaiser,
     'Cumulative variance': f"{cum_var[n_kaiser-1]*100:.1f}%"},
]
for label, n in threshold_counts.items():
    summary_rows.append({'Criterion': label, 'N components': n,
                         'Cumulative variance': f"{cum_var[n-1]*100:.1f}%"})

print('FACTOR COUNT SUMMARY')
print('=' * 55)
print(f'  Total series      : {df_clean.shape[1]}')
print(f'  Observations      : {df_clean.shape[0]}')
print(f'  Total PCs         : {n_components}')
print(f'  PC1 variance      : {var_ratio[0]*100:.1f}%')
print()
print(pd.DataFrame(summary_rows).to_string(index=False))
print('=' * 55)

## Step 6 — Top Factor Loadings

Each loading shows how strongly an original variable contributes to that component.

In [ ]:
n_pcs   = min(3, n_components)
top_n   = CONFIG['TOP_LOADINGS']
col_names = list(df_clean.columns)

fig, axes = plt.subplots(1, n_pcs, figsize=(6 * n_pcs, 6))
if n_pcs == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    loadings = pd.Series(pca.components_[i], index=col_names)
    top      = loadings.abs().nlargest(top_n)
    vals     = loadings[top.index]
    colors   = ['steelblue' if v > 0 else 'tomato' for v in vals]
    ax.barh(range(top_n), vals.values, color=colors, alpha=0.8)
    ax.set_yticks(range(top_n)); ax.set_yticklabels(vals.index, fontsize=9)
    ax.invert_yaxis(); ax.axvline(x=0, color='black', linewidth=0.5)
    ax.set(xlabel='Loading', title=f'PC{i+1} ({var_ratio[i]*100:.1f}% variance)')
    ax.grid(True, alpha=0.3, axis='x')

plt.suptitle(f'Top {top_n} Loadings per Component', y=1.02)
plt.tight_layout(); plt.show()

## Step 7 — Factor Time Series & Sharpe Ratios

Project the original data onto the principal components to get factor scores over time.

In [ ]:
n_factors = min(5, n_components)
factor_scores = pd.DataFrame(
    X_scaled @ pca.components_[:n_factors].T,
    index   = df_clean.index,
    columns = [f'PC{i+1}' for i in range(n_factors)],
)

# Sharpe ratios (treating factor scores as return-like series)
sharpe_rows = []
for col in factor_scores.columns:
    s = factor_scores[col]
    sharpe = (s.mean() / s.std(ddof=1)) * np.sqrt(12)
    sharpe_rows.append({'Factor': col, 'Mean': s.mean(), 'Std': s.std(ddof=1), 'Sharpe (ann.)': sharpe})

print('Factor Score Statistics:')
print(pd.DataFrame(sharpe_rows).to_string(index=False, float_format='{:.4f}'.format))

# Cumulative factor score plot
fig, ax = plt.subplots(figsize=(12, 5))
for col in factor_scores.columns:
    cumulative = (1 + factor_scores[col] / 100).cumprod()
    ax.plot(factor_scores.index, cumulative.values, label=col, linewidth=1.5)
ax.axhline(y=1, color='gray', linestyle='--', linewidth=0.8)
ax.set(xlabel='Date', ylabel='Cumulative (growth of 1)', title='Cumulative Factor Scores (PC1–PC5)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Step 8 — Save Results

In [ ]:
if CONFIG['SAVE_RESULTS']:
    os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)

    # Eigenvalue table
    ev_df = pd.DataFrame({
        'component': [f'PC{i+1}' for i in range(n_components)],
        'eigenvalue': eigenvalues,
        'var_pct': var_ratio * 100,
        'cum_var_pct': cum_var * 100,
    })
    path = os.path.join(CONFIG['OUTPUT_DIR'], 'pca_eigenvalues.csv')
    ev_df.to_csv(path, index=False)
    print(f'Saved: {path}')

    # Factor count summary
    path = os.path.join(CONFIG['OUTPUT_DIR'], 'pca_factor_summary.csv')
    pd.DataFrame(summary_rows).to_csv(path, index=False)
    print(f'Saved: {path}')

    # Loadings matrix
    loadings_df = pd.DataFrame(
        pca.components_[:n_factors].T,
        index=col_names,
        columns=[f'PC{i+1}' for i in range(n_factors)],
    )
    path = os.path.join(CONFIG['OUTPUT_DIR'], 'pca_loadings.csv')
    loadings_df.to_csv(path)
    print(f'Saved: {path}')
else:
    print('SAVE_RESULTS = False — skipping.')